In [ ]:
import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from tabulate import tabulate

In [3]:
log = pm4py.read_xes("../logs/event-log.xes")

/Users/bd/Projects/thesis/pm4py/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
/Users/bd/Projects/thesis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 13087/13087 [00:00<00:00, 25910.28it/s]


In [4]:
df = log_converter.apply(log, variant=log_converter.Variants.TO_DATA_FRAME)

In [5]:
df.head(10)

,case:REG_DATE,org:resource,concept:name,lifecycle:transition,case:AMOUNT_REQ,time:timestamp,case:concept:name
0,2011-10-01 02:38:00+00:00,112,submitted,COMPLETE,20000,2011-10-01 02:38:00+00:00,173688
1,2011-10-01 02:38:00+00:00,112,pre-accepted,COMPLETE,20000,2011-10-01 02:39:00+00:00,173688
2,2011-10-01 02:38:00+00:00,10862,accepted,COMPLETE,20000,2011-10-01 13:42:00+00:00,173688
3,2011-10-01 02:38:00+00:00,10862,finalized,COMPLETE,20000,2011-10-01 13:45:00+00:00,173688
4,2011-10-01 02:38:00+00:00,10629,registered,COMPLETE,20000,2011-10-13 12:37:00+00:00,173688
5,2011-10-01 02:38:00+00:00,10629,approved,COMPLETE,20000,2011-10-13 12:37:00+00:00,173688
6,2011-10-01 02:38:00+00:00,10629,activated,COMPLETE,20000,2011-10-13 12:37:00+00:00,173688
7,2011-10-01 10:08:00+00:00,112,submitted,COMPLETE,5000,2011-10-01 10:08:00+00:00,173691
8,2011-10-01 10:08:00+00:00,112,pre-accepted,COMPLETE,5000,2011-10-01 10:09:00+00:00,173691
9,2011-10-01 10:08:00+00:00,10862,accepted,COMPLETE,5000,2011-10-01 16:33:00+00:00,173691


In [6]:
import xml.etree.ElementTree as ET
import pandas as pd

def tabulate_event_log(xes_path: str) -> pd.DataFrame:
    """
    Reads an XES (.xes) event log file and returns a pandas DataFrame
    with one row per event, including the case ID and all event attributes.
    """
    tree = ET.parse(xes_path)
    root = tree.getroot()

    # Detect namespace if present
    ns = ''
    if root.tag.startswith('{'):
        uri = root.tag.split('}')[0].strip('{')
        ns = f'{{{uri}}}'

    records = []
    # Iterate through each trace (case)
    for trace in root.findall(f'.//{ns}trace'):
        # Extract case ID (concept:name) at trace level
        case_id = None
        for attr in trace.findall(f'{ns}string'):
            if attr.attrib.get('key') == 'concept:name':
                case_id = attr.attrib.get('value')
                break

        # Iterate through events in the trace
        for event in trace.findall(f'{ns}event'):
            rec = {'case_id': case_id}
            # Extract all attributes in the event
            for attr in event:
                key = attr.attrib.get('key')
                val = attr.attrib.get('value')
                rec[key] = val
            records.append(rec)

    # Build DataFrame
    df = pd.DataFrame(records)

    # Convert timestamp column to datetime if present
    if 'time:timestamp' in df.columns:
        df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

    # Optionally sort by case_id and timestamp
    if 'time:timestamp' in df.columns:
        df = df.sort_values(['case_id', 'time:timestamp'])

    return df

In [7]:
df = tabulate_event_log('event-log.xes')
df.head(50)

FileNotFoundError: [Errno 2] No such file or directory: 'event-log.xes'